# Lab 11 — Self-Training, Label Spreading and Autoencoders, Built from Scratch

**Class 11.** Guide: `practice/11_autoencoders.md`. Lecture: `slides/11_autoencoders.pdf`.
Demos: `01_autoencoder.ipynb` to `07_pretraining.ipynb`. In class (60 minutes): Parts A to D; at home: Parts E and F.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready:
it draws the plots and prints the comparisons once your functions work.

**The rule of this lab.** Each idea of the class is one small function: the pseudo-label loop, label spreading, the gradient of a linear autoencoder,
the threshold of a detector. You write it with NumPy and compare it with its library twin (scikit-learn, PCA, Keras). The notebook prints the difference; explain it.


In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import warnings

import numpy as np
import polars as pl
from lab11_utils import (
    classifier_from,
    few_labels,
    fit_ae,
    knn_affinity,
    make_ae,
    plot_curve,
    plot_digits,
    recon_error,
    split,
)
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
xtr, ytr, xte, yte = split()
lab = few_labels(ytr, 5, seed=0)
unl = np.setdiff1d(np.arange(len(xtr)), lab)
print(f"{len(xtr)} train digits ({len(lab)} labelled, {len(unl)} unlabelled), {len(xte)} test digits")

# Part A — Self-Training (12 min)

The model labels its own data: fit on the labelled rows, give the unlabelled rows whose top probability is at least $\tau$ their most probable class,
add them to the labelled set and repeat.

## A1. The pseudo-label loop

Implement `self_train(xl, yl, xu, tau, rounds)`: a `LogisticRegression(C=10, max_iter=3000)` is fitted in every round;
return the final model and the list with the number of rows added in each round. Stop when no row passes the threshold.

In [ ]:
def self_train(xl, yl, xu, tau=0.9, rounds=10):
    raise NotImplementedError

The driver compares three models on the test digits: the model fitted on the 50 labels only, your loop, and scikit-learn's `SelfTrainingClassifier` with the same $\tau$.

In [ ]:
base = LogisticRegression(C=10, max_iter=3000).fit(xtr[lab], ytr[lab])
model, added = self_train(xtr[lab], ytr[lab], xtr[unl], tau=0.9)
y_semi = ytr.copy()
y_semi[unl] = -1
twin = SelfTrainingClassifier(LogisticRegression(C=10, max_iter=3000), threshold=0.9, max_iter=10).fit(xtr, y_semi)
print(
    pl.DataFrame(
        {
            "model": ["50 labels only", "our self-training", "SelfTrainingClassifier"],
            "test accuracy": [base.score(xte, yte), model.score(xte, yte), twin.score(xte, yte)],
        }
    )
)
print("rows added per round:", added)
print(f"agreement of ours with the twin on the test digits: {np.mean(model.predict(xte) == twin.predict(xte)):.3f}")

## A2. The threshold

The cell sweeps $\tau$ and prints, for each value, the share of unlabelled rows that became pseudo-labelled and the accuracy of those pseudo-labels.

In [ ]:
rows = []
for tau in (0.5, 0.7, 0.9, 0.99):
    m, add = self_train(xtr[lab], ytr[lab], xtr[unl], tau=tau)
    rows.append((tau, sum(add) / len(unl), m.score(xte, yte)))
print(pl.DataFrame(rows, schema=["tau", "share added", "test accuracy"], orient="row"))

**Questions A.**
1. Which $\tau$ is best, and why does a small $\tau$ not help more? Name the failure that makes the pseudo-label accuracy fall in later rounds.
2. Why does our accuracy differ from the twin's in the third decimal? (Hint: how many rows does each round add?)
3. The 50 labels are changed: five of them get a wrong class. What does self-training do with the error?

# Part B — Label Spreading (14 min)

Rows are nodes of a graph; `knn_affinity(X, k)` gives the affinity matrix $W$ ($n \times n$, Gaussian weights on the $k$ nearest neighbours).
The labels flow along the edges: $F = (I - \alpha S)^{-1} Y_0$ with $S = D^{-1/2} W D^{-1/2}$ and $Y_0$ the one-hot labels of the labelled nodes.

## B1. Normalise and solve

Implement `spread(W, y, labelled, alpha)`: build $S$ (a degree of 0 counts as 1), build $Y_0$ ($n \times 10$), solve the linear system with `np.linalg.solve`
(never invert the matrix) and return the class of every node.

In [ ]:
def spread(W, y, labelled, alpha=0.9):
    raise NotImplementedError

The driver puts all 1797 digits in one graph (the test digits are unlabelled nodes) and scores the unlabelled nodes. Twin: `LabelSpreading(kernel="knn")`.

In [ ]:
Xall = np.vstack([xtr, xte])
yall = np.concatenate([ytr, yte])
labelled = few_labels(yall, 5, seed=0)
rest = np.setdiff1d(np.arange(len(yall)), labelled)
W = knn_affinity(Xall, 10)
pred = spread(W, yall, labelled, 0.9)
ys = yall.copy()
ys[rest] = -1
twin = LabelSpreading(kernel="knn", n_neighbors=10, alpha=0.9, max_iter=200).fit(Xall, ys)
lr = LogisticRegression(C=10, max_iter=3000).fit(Xall[labelled], yall[labelled])
print(
    pl.DataFrame(
        {
            "method": ["50 labels only", "our spreading", "LabelSpreading"],
            "accuracy on unlabelled": [
                lr.score(Xall[rest], yall[rest]),
                np.mean(pred[rest] == yall[rest]),
                np.mean(twin.transduction_[rest] == yall[rest]),
            ],
        }
    )
)

## B2. The value of $\alpha$

The cell sweeps $\alpha$ on your function.

In [ ]:
rows = [(a, float(np.mean(spread(W, yall, labelled, a)[rest] == yall[rest]))) for a in (0.1, 0.5, 0.9, 0.99)]
print(pl.DataFrame(rows, schema=["alpha", "accuracy on unlabelled"], orient="row"))

**Questions B.**
1. What happens at $\alpha \to 0$? Which classifier do you get?
2. Why does the graph method beat the pseudo-label loop on the digits, and on which kind of data would it fail?
3. Your accuracy and the twin's differ. Which choice of the graph explains it?

# Part C — A Linear Autoencoder by Gradient Descent (16 min)

A linear autoencoder rebuilds $x$ from $z = E x$ with $\hat{x} = D z$. In rows: $Z = X E^\top$, $\hat{X} = Z D^\top$, loss $L = \frac{1}{n} \lVert \hat{X} - X \rVert_F^2$.
With the residual $R = \hat{X} - X$ the gradients are $\partial L / \partial D = \frac{2}{n} R^\top Z$ and $\partial L / \partial E = \frac{2}{n} (R D)^\top X$.

## C1. The gradient step

Implement `linear_ae(X, k, lr, steps, seed)`: the digits are centred first (subtract the mean); start from small random $E$ ($k \times d$) and $D$ ($d \times k$),
take `steps` gradient steps, return $E$, $D$ and the list of losses.

In [ ]:
def linear_ae(X, k, lr=0.05, steps=2000, seed=0):
    raise NotImplementedError

The driver trains your autoencoder with $k = 5$ and compares the test error per pixel with PCA ($k = 5$) and with the same-size Keras linear network is left to the demo.
Principal angle: how far the span of your decoder is from the PCA subspace.

In [ ]:
from scipy.linalg import subspace_angles

mu = xtr.mean(0)
Xc, Xtc = xtr - mu, xte - mu
E, D, losses = linear_ae(Xc, 5)
pca = PCA(5).fit(xtr)
err_ours = float(np.mean((Xtc @ E.T @ D.T - Xtc) ** 2))
err_pca = float(np.mean((pca.inverse_transform(pca.transform(xte)) - xte) ** 2))
print(pl.DataFrame({"model": ["our linear AE", "PCA (k = 5)"], "test error per pixel": [err_ours, err_pca]}))
print(f"loss at step 0: {losses[0]:.3f}, at the end: {losses[-1]:.3f}")
print(
    f"largest principal angle between our decoder and the PCA subspace: {np.degrees(subspace_angles(D, pca.components_.T).max()):.1f} degrees"
)

The ReLU network of the demos with the same code size, for question 3.

In [ ]:
relu_ae, _ = make_ae(64, 5, hidden=(32,))
fit_ae(relu_ae, xtr, epochs=100)
err_relu = float(np.mean((np.asarray(relu_ae.predict(xte, verbose=0)) - xte) ** 2))
pca5 = PCA(5).fit(xtr)
err_pca5 = float(np.mean((pca5.inverse_transform(pca5.transform(xte)) - xte) ** 2))
print(
    pl.DataFrame(
        {
            "model": ["PCA (k = 5)", "ReLU AE 64-32-5-32-64"],
            "test error per pixel": [err_pca5, err_relu],
            "weights": [64 * 5 + 5 * 64 + 64 + 5, relu_ae.count_params()],
        }
    )
)

**Questions C.**
1. The two test errors: are they equal? Why does a network with no non-linearity learn the PCA subspace?
2. Is the angle near 0? Which property of the singular values makes it large even when the errors agree?
3. Compare the ReLU network with PCA at the same $k = 5$: which rebuilds the digits better, which has more weights, and what would you check before calling the ReLU network better?

# Part D — The Reconstruction Error as an Anomaly Score (12 min)

Normal rows are the digits 0 to 7; the anomalies are 8 and 9. The autoencoder sees only normal rows (80 % to fit, 20 % to set the threshold).
The score of a row is its reconstruction error; an alarm is raised above the threshold $\theta$.

## D1. The threshold and the report

Implement `report(e_val, e_test, is_anomaly, q)`: $\theta$ is the quantile $q$ of the errors `e_val` of the held-out normal rows. Return a dict with
`theta`, `recall` (share of anomalies above $\theta$), `false_alarm` (share of normal test rows above $\theta$) and `auroc` (`roc_auc_score(is_anomaly, e_test)`).

In [ ]:
def report(e_val, e_test, is_anomaly, q=0.95):
    raise NotImplementedError

The driver trains the autoencoder ($64 \to 32 \to 4 \to 32 \to 64$) on the normal rows and compares it with a PCA detector of the same code size.

In [ ]:
normal = ytr < 8
xn = xtr[normal]
cut = int(0.8 * len(xn))
xfit, xval = xn[:cut], xn[cut:]
is_anom = yte >= 8
ae, _ = make_ae(64, 4, hidden=(32,))
fit_ae(ae, xfit, epochs=150)
pca4 = PCA(4).fit(xfit)
pca_err = lambda X: np.mean((pca4.inverse_transform(pca4.transform(X)) - X) ** 2, axis=1)  # noqa: E731
rows = []
for name, ev, et in (
    ("autoencoder", recon_error(ae, xval), recon_error(ae, xte)),
    ("PCA", pca_err(xval), pca_err(xte)),
):
    rows.append({"detector": name, **report(ev, et, is_anom, 0.95)})
print(pl.DataFrame(rows))

## D2. The quantile

Sweep $q$ for the autoencoder: a lower quantile raises recall and the false alarms together.

In [ ]:
ev, et = recon_error(ae, xval), recon_error(ae, xte)
print(pl.DataFrame([{"q": q, **report(ev, et, is_anom, q)} for q in (0.5, 0.8, 0.95, 0.99)]))

**Questions D.**
1. What does the AUROC say that recall and false alarms do not? Why does it not change with $q$?
2. Which detector wins at code size 4, and which would you try at code size 64?
3. The threshold comes from validation rows of the normal class only. What would go wrong if it came from the training rows the autoencoder was fitted on?

# Part E — Pretraining with Few Labels (home)

## E1. A probe on the code

Implement `probe(encoder, x_labelled, y_labelled, x_test, y_test)`: encode with `encoder.predict(x, verbose=0)`, fit a `LogisticRegression(C=10, max_iter=3000)` on the codes of the labelled rows and return the test accuracy.

In [ ]:
def probe(encoder, xl, yl, xt, yt):
    raise NotImplementedError

The driver compares a random encoder, an encoder pretrained on the 1257 unlabelled training digits (denoising, noise 0.1), pixels, and the fine-tuned classifier, for 1, 5 and 20 labels per class (2 draws).

In [ ]:
ae16, enc16 = make_ae(64, 16, hidden=(32,))
fit_ae(ae16, xtr, noise=0.1, epochs=100)
_, enc_rand = make_ae(64, 16, hidden=(32,))
rows = []
for per in (1, 5, 20):
    acc = {"pixels": [], "random encoder": [], "pretrained": [], "pretrained, fine-tuned": []}
    for seed in range(2):
        idx = few_labels(ytr, per, seed)
        acc["pixels"].append(LogisticRegression(C=10, max_iter=3000).fit(xtr[idx], ytr[idx]).score(xte, yte))
        acc["random encoder"].append(probe(enc_rand, xtr[idx], ytr[idx], xte, yte))
        acc["pretrained"].append(probe(enc16, xtr[idx], ytr[idx], xte, yte))
        m = classifier_from(enc16, seed=seed)
        m.fit(xtr[idx], ytr[idx], epochs=100, batch_size=16, verbose=0)
        acc["pretrained, fine-tuned"].append(float(m.evaluate(xte, yte, verbose=0)[1]))
    rows.append({"per class": per, **{k: float(np.mean(v)) for k, v in acc.items()}})
res = pl.DataFrame(rows)
print(res)

**Questions E.**
1. How much does the pretraining add to the random encoder? Does it beat the pixels? Say why not, or why.
2. When would you expect pretraining to pay: which data and how many unlabelled rows?

# Part F — Denoising (home)

## F1. Clean the digits

Implement `noisy(X, sd, seed)`: Gaussian noise of standard deviation `sd` added to every pixel, then clipped to $[0, 1]$ (`np.clip`), as `float32`.

In [ ]:
def noisy(X, sd=0.3, seed=0):
    raise NotImplementedError

The driver trains a plain and a denoising autoencoder ($64 \to 32 \to 8 \to 32 \to 64$) and shows the first eight test digits clean, noisy and cleaned.

In [ ]:
xt_noisy = noisy(xte, 0.3, seed=1)
plain, _ = make_ae(64, 8, hidden=(32,))
fit_ae(plain, xtr, epochs=100)
den, _ = make_ae(64, 8, hidden=(32,))
fit_ae(den, xtr, noise=0.3, epochs=100)
err = lambda m: float(np.mean((np.asarray(m.predict(xt_noisy, verbose=0)) - xte) ** 2))  # noqa: E731
print(
    pl.DataFrame(
        {
            "model": ["noisy input itself", "plain AE", "denoising AE"],
            "error against the clean digit": [float(np.mean((xt_noisy - xte) ** 2)), err(plain), err(den)],
        }
    )
)
plot_digits(
    {
        "clean": xte[:8],
        "noisy": xt_noisy[:8],
        "plain": np.asarray(plain.predict(xt_noisy[:8], verbose=0)),
        "denoising": np.asarray(den.predict(xt_noisy[:8], verbose=0)),
    }
)

**Questions F.**
1. Why does the plain autoencoder not remove the noise, while the denoising one does?
2. Which setting of `noise` in training would you pick, and what happens if it is much larger than in the test?